# ML Assignment 2 — Dry Bean Classification

This notebook is designed for the assignment requirements:

- Dataset: **Dry Bean Dataset**
- Classification models:
  1. Logistic Regression
  2. Decision Tree
  3. K-Nearest Neighbors (KNN)
  4. Gaussian Naive Bayes
  5. Random Forest
- Evaluation metrics for each model:
  - Accuracy
  - AUC
  - Precision
  - Recall
  - F1 Score
  - Matthews Correlation Coefficient (MCC)

> Place `Dry_Bean_Dataset.xlsx` in the same folder as this notebook before running it.


## 1. Import required libraries

If a package is missing in your environment, install it before continuing.  
For Excel support, `openpyxl` may be required.


In [ ]:
# Uncomment if openpyxl is missing:
# !pip install openpyxl

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

print("Libraries imported successfully!")


## 2. Load the Dry Bean dataset

In [ ]:
DATA_FILE = "Dry_Bean_Dataset.xlsx"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. "
        "Place the Excel file in the same folder as this notebook."
    )

df = pd.read_excel(DATA_FILE)
print("Dataset loaded successfully!")


## 3. Inspect the dataset

In [ ]:
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
df.info()


In [ ]:
print("Total missing values:", df.isnull().sum().sum())
print("\nMissing values by column:")
display(df.isnull().sum().to_frame("Missing Values"))


In [ ]:
print("Number of classes:", df["Class"].nunique())
print("\nClass distribution:")
display(df["Class"].value_counts().to_frame("Count"))


In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)


## 4. Data cleaning

Duplicate records can cause the same observation to appear more than once and may bias model evaluation.  
This notebook removes exact duplicate rows before the train/test split.

If your instructor asks you to preserve duplicates, comment out the `drop_duplicates()` line.


In [ ]:
rows_before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)

print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal :", rows_after)
print("Duplicates removed           :", rows_before - rows_after)


## 5. Separate input features and target

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

print("Number of input features:", X.shape[1])
print("Target column:", y.name)
print("Target classes:", sorted(y.unique()))


## 6. Create a stratified train/test split

- 80% training data
- 20% test data
- `random_state=42` makes the split reproducible.
- `stratify=y` helps preserve the class proportions in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples    :", X_test.shape[0])


## 7. Define the five required models

`StandardScaler` is used inside the pipelines for Logistic Regression and KNN because these models are sensitive to feature scale.

Decision Tree, Gaussian Naive Bayes, and Random Forest are trained on the original feature values.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "Naive Bayes": GaussianNB(),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

print("Models defined:")
for name in models:
    print("-", name)


## 8. Train and evaluate all five models

For multiclass classification:

- Precision, Recall, and F1 use **weighted averaging**.
- AUC uses **One-vs-Rest (OvR)** with weighted averaging.
- MCC supports multiclass classification directly.


In [ ]:
results = []

for model_name, model in models.items():
    print(f"Training {model_name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    # The order of probability columns follows model.classes_
    auc = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr",
        average="weighted",
        labels=model.classes_
    )

    results.append({
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": auc,
        "Precision": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

print("Training and evaluation completed.")


## 9. Model comparison table

In [ ]:
results_df = pd.DataFrame(results)

metric_columns = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
results_df[metric_columns] = results_df[metric_columns].round(4)

display(results_df)


In [ ]:
best_model_name = results_df.loc[results_df["F1"].idxmax(), "ML Model Name"]
print("Best model based on weighted F1 Score:", best_model_name)


## 10. Detailed report for each model

This section prints the classification report for each trained model.


In [ ]:
for model_name, model in models.items():
    print("=" * 80)
    print(model_name)
    print("=" * 80)

    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))


## 11. Confusion matrix for each model

In [ ]:
for model_name, model in models.items():
    y_pred = model.predict(X_test)

    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        xticks_rotation=45,
        cmap="Blues",
        ax=ax
    )
    ax.set_title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()


## 12. Save test data

The Streamlit app can use this held-out test set for model evaluation.  
The true `Class` label is included so the app can calculate metrics and display a confusion matrix.


In [ ]:
test_data = X_test.copy()
test_data["Class"] = y_test.values

test_data.to_csv("test_data.csv", index=False)

print("Saved: test_data.csv")
print("Shape:", test_data.shape)
display(test_data.head())


## 13. Save trained models

In [ ]:
os.makedirs("model", exist_ok=True)

model_filenames = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "KNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest": "random_forest.pkl"
}

for model_name, filename in model_filenames.items():
    path = os.path.join("model", filename)
    joblib.dump(models[model_name], path)
    print("Saved:", path)


## 14. Save comparison results

In [ ]:
results_df.to_csv("model_comparison.csv", index=False)
print("Saved: model_comparison.csv")


## 15. Observations — complete after running the models

Use your actual results from the comparison table above. Do **not** copy generic observations without checking the measured values.

| Model | Observation |
|---|---|
| Logistic Regression | Add your observation based on Accuracy, AUC, Precision, Recall, F1 and MCC. |
| Decision Tree | Add your observation based on the measured results. |
| KNN | Add your observation based on the measured results. |
| Naive Bayes | Add your observation based on the measured results. |
| Random Forest | Add your observation based on the measured results. |
| Overall Winner | Select the model that performs best overall and explain why. |

When we have your actual output, we can write these observations together.


## 16. Files to keep for the final project

After successfully running this notebook, your working folder should contain files similar to:

```text
ML_Assignment_2_DryBean/
│
├── Dry_Bean_Dataset.xlsx
├── Dry_Bean_ML_Assignment.ipynb
├── test_data.csv
├── model_comparison.csv
└── model/
    ├── logistic_regression.pkl
    ├── decision_tree.pkl
    ├── knn.pkl
    ├── naive_bayes.pkl
    └── random_forest.pkl
```

Later we will add:

```text
app.py
requirements.txt
README.md
```

and then deploy the Streamlit application.
